# 06 Bericht und AAVSO-Export

**Eingabe**: `zeitreihe.csv` aus NB05 (bei Farbkameras optional `zeitreihe_transformiert.csv`), Plan aus NB00, Beobachter aus NB01, `field_config.json` aus NB03  
**Ausgabe**: Datei im **AAVSO Extended Format** (`PHOTOMETRIE_PY_DATA/exports/`), Prüfbericht, Zusammenfassung. **Kein automatischer Upload** — die Datei wird von Hand über WebObs eingereicht.

**Regeln** (AAVSO Extended File Format v1.2 und DSLR-Handbuch §6.7, im Wortlaut gelesen): Datum = Julianisches Datum der **Mitte** der Belichtung in UT (`jd_mid`; BJD_TDB ist im Format nicht vorgesehen); Filter `V`/`B`/`R` nur mit `TRANS=YES`, wenn transformiert,
sonst bei Farbkameras `TG`/`TB`/`TR` mit `TRANS=NO` (Comps im jeweiligen Band); `ENSEMBLE` als Comp-Name mit `CMAG=na`; Check-Magnitude bei transformierten Werten die **transformierte**; `OBSTYPE` `CCD` oder `DSLR` (Farbkameras: vorläufig `DSLR`, bei der AAVSO bestätigen).
Eigene Comps (nicht AAVSO-Sequenz, z. B. Gaia-abgeleitet): `CHART` = Name der Quelle, Details in `NOTES`; **für Gaia-abgeleitete Werte gilt: nur Notbehelf** (KONZEPT §11.7).

## Schritt 1 — Eingaben laden

In [ ]:
import os, json, datetime
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

from varphot import (TransformSolution, build_extended, notes_string, observations_from_lightcurve, validate_extended, load_plan, __version__)
from varphot.observer import data_dir, load_config, all_sites

TAG = os.environ.get("VARPHOT_TAG", "delta_cep_lco_0m4_11")
RESULTS = (data_dir() if data_dir().is_dir() else Path("../data")) / "results" / TAG
EXPORTS = (data_dir() if data_dir().is_dir() else Path("../data")) / "exports"
LC = pd.read_csv(RESULTS / "zeitreihe.csv")
LC_TR = pd.read_csv(RESULTS / "zeitreihe_transformiert.csv") if (RESULTS / "zeitreihe_transformiert.csv").exists() else None
PLAN = load_plan(os.environ["VARPHOT_PLAN"]) if os.environ.get("VARPHOT_PLAN") else None
OBS, _own = load_config()
FIELD_CONFIG = Path(os.environ.get("VARPHOT_FIELD_CONFIG", "../data/fields/del_Cep_lco_0m4_field_config.json"))
FC = json.loads(FIELD_CONFIG.read_text()) if FIELD_CONFIG.exists() else None
TRANSFORM_FILE = os.environ.get("VARPHOT_TRANSFORM_FILE")
SOL = TransformSolution.load(TRANSFORM_FILE) if TRANSFORM_FILE else None

# Herkunft der Daten: Testdaten fremder Beobachter dürfen NIE unter dem eigenen Code eingereicht werden
IS_TESTDATA = TAG.startswith("delta_cep_lco") and not os.environ.get("VARPHOT_NB06_REAL")
OBSCODE = "TST01" if IS_TESTDATA else (OBS.aavso_code or "")
STAR = os.environ.get("VARPHOT_STAR") or (PLAN.target if PLAN else (FC["ziel"]["name"].replace("* ", "") if FC else "DEL CEP")).upper()
print(f"Zeitreihe {len(LC)} Punkte" + (f", transformiert {len(LC_TR)} Punkte" if LC_TR is not None else "") + f"; Stern {STAR}; Beobachtercode {OBSCODE or '— fehlt (NB01) —'}")
if IS_TESTDATA:
    print("⚠ TESTDATEN (LCO-Archiv, nicht deine Beobachtung): Code TST01 und Dateiname markiert — nicht einreichen. Eigene Daten: VARPHOT_NB06_REAL=1.")

## Schritt 2 — Band, Filtercode und `TRANS` festlegen

* **Untransformiert** (`mag` aus NB05): Mono-Kamera mit V-Filter → `V`, `TRANS=NO` (Handbuch/Format: Standardfilter, Differenzphotometrie gegen V-Comps); Farbkamera/RGB-Filter → `TG`/`TB`/`TR`, `TRANS=NO`.
* **Transformiert** (`zeitreihe_transformiert.csv` aus NB05 + Koeffizienten aus NB08): `V`/`B`/`R`, `TRANS=YES`, die Koeffizienten stehen in `NOTES`. Nur für Bänder, deren Gütetest in NB08 bestanden wurde.

Auswahl per Umgebungsvariable `VARPHOT_NB06_MODE=transformiert`; sonst untransformiert.

In [ ]:
MODE_T = os.environ.get("VARPHOT_NB06_MODE", "untransformiert") == "transformiert"
OBSTYPE = os.environ.get("VARPHOT_OBSTYPE") or (PLAN.obstype if PLAN else "CCD")
COLOUR = bool(PLAN and PLAN.method in ("osc", "dslr", "monorgb"))
OBSTYPE_UNCONFIRMED = COLOUR and not os.environ.get("VARPHOT_OBSTYPE_CONFIRMED")
if OBSTYPE_UNCONFIRMED:
    print("ENTSCHEIDUNG nötig — OBSTYPE bei Farbkamera/RGB-Filter: " + OBSTYPE + " (Vorgabe aus dem Plan).\n"
          "  Lesart A (AAVSO-Forum, Hawkins): 'DSLR' gilt für DSLR, Farb-CMOS (OSC) und Mono-CMOS mit RGB-Filtern.\n"
          "  Lesart B (Format-Seite): 'CMOS → CCD'.\n"
          "  Ändern: VARPHOT_OBSTYPE=CCD|DSLR; nach Klärung bei der AAVSO: VARPHOT_OBSTYPE_CONFIRMED=1 (dann entfällt dieser Hinweis).")
if MODE_T:
    if LC_TR is None or SOL is None:
        raise RuntimeError("transformierter Export braucht zeitreihe_transformiert.csv (NB05) und VARPHOT_TRANSFORM_FILE (NB08)")
    BAND = os.environ.get("VARPHOT_NB06_BAND", "V")
    ok_band = SOL.validation.get(BAND, {})
    if not ok_band or ok_band.get("rms_transformed", 1) > ok_band.get("rms_standardized", 0) or ok_band.get("rms_transformed", 1) > 0.06:
        raise RuntimeError(f"Band {BAND}: Gütetest in NB08 nicht bestanden (Transformation verbessert nicht oder rms > 0,06 mag) — als TG/TB/TR untransformiert melden")
    FILT, TRANS = BAND, True
    DATA = LC_TR.rename(columns={BAND: "mag", "V_err": "mag_err"})[["jd_mid", "mag", "mag_err", "airmass"] + (["V_check"] if (BAND == "V" and "V_check" in LC_TR) else [])]
    if "V_check" in DATA:
        DATA = DATA.rename(columns={"V_check": "check_mag"})                  # transformierte Check-Magnitude (Handbuch §6.7)
    COEF = dict(Tv_bv=round(SOL.Tv_bv, 4) if BAND == "V" else None, Tbv=round(SOL.Tbv, 4))
else:
    FILT = PLAN.aavso_filter if PLAN else "V"
    TRANS = False
    DATA = LC
    COEF = {}
print(f"Meldung: Filter {FILT}, TRANS={'YES' if TRANS else 'NO'}, OBSTYPE {OBSTYPE}, {len(DATA)} Punkte")

## Schritt 3 — Vergleichssterne, Chart und Check-Stern

* Stammen **alle** Comps aus der AAVSO-Sequenz, trägt `CHART` die **Sequenz-ID** (auf der VSP-Seite fett über der Fototabelle: „Report this sequence as … in the chart field“; trage sie unten ein).
* Sind **Gaia-abgeleitete** Comps dabei, ist die Sequenz-ID falsch: `CHART` = Name der Quelle, die Comps stehen in `NOTES` (Format-Seite: „comment code K“ für Nicht-AAVSO-Charts).
* Der **Check-Stern** steht in `KNAME`/`KMAG`; bei transformierten Werten gehört die transformierte Magnitude hinein (hier nicht verfügbar → `na`).

In [ ]:
CHART_ID = os.environ.get("VARPHOT_CHART_ID") or ((FC or {}).get("vsp", {}).get("chartid") or "")   # Sequenz-ID: aus NB03 (VSP-Antwort), sonst von Hand
comps = [s for s in (FC["sterne"] if FC else []) if s["rolle"] == "comp"]
check = next((s for s in (FC["sterne"] if FC else []) if s["rolle"] == "check"), None)
n_gaia = sum(str(s["auid"]).startswith("Gaia") for s in comps)
if not comps:
    CHART, WARN_CH = "na", "keine field_config.json: Comps unbekannt"
elif n_gaia:
    CHART, WARN_CH = "Gaia DR3 V-deriv", f"{n_gaia} von {len(comps)} Comps Gaia-abgeleitet → keine AAVSO-Sequenz-ID; als Notbehelf kennzeichnen"
elif CHART_ID:
    CHART, WARN_CH = CHART_ID, ""
else:
    CHART, WARN_CH = "na", "CHART_ID fehlt: field_config.json ohne Sequenz-ID (älter als NB03 vom 2026-11-10?) — NB03 neu ausführen oder VARPHOT_CHART_ID setzen"
CHECK_LABEL = (str(check["label"]) if check else os.environ.get("VARPHOT_CHECK_LABEL") or ("g1" if TAG.startswith("delta_cep_lco") else None))
KNAME = CHECK_LABEL if (not MODE_T or "check_mag" in DATA) else None
print(f"Comps: {len(comps)} (davon Gaia-abgeleitet: {n_gaia}), Check-Stern: {KNAME or '—'}, CHART: {CHART}")
if WARN_CH:
    print("⚠", WARN_CH)

## Schritt 4 — Datei erzeugen und prüfen

Die Prüfung folgt den Regeln des Formats (Pflichtparameter, Feldzahl und -längen, Filtercodes, Ensemble-Regel, JD, Leerzeilen …). Sie ersetzt **nicht** die Prüfung durch WebObs (Sternname, Sequenz-ID).

In [ ]:
core_note = "Ziel gesaettigt, Kernmaske-Korrektur" if ("core_corrected" in LC and bool(LC["core_corrected"].any()) and not MODE_T) else ""
NOTES = notes_string("varphot %s; Ensemble %d Comps%s%s" % (__version__, int(np.nanmedian(LC["n_comps"])) if "n_comps" in LC else 0,
                     "; " + core_note if core_note else "", "; Comps teils Gaia-abgeleitet (K)" if n_gaia else ""), **COEF)
OBSV = observations_from_lightcurve(DATA, STAR, FILT, TRANS, chart=CHART, kname=KNAME, kmag_col=("check_mag" if (KNAME and "check_mag" in DATA) else None),
                                    notes=NOTES)
TEXT = build_extended(OBSV, OBSCODE or "XXXXX", f"varphot {__version__}", OBSTYPE, comments=[f"Quelle: {TAG}", "Zeit: JD (UT) Mitte der Belichtung; BJD_TDB steht in zeitreihe.csv"] + (["TESTDATEN - NICHT EINREICHEN"] if IS_TESTDATA else []))
ISSUES = validate_extended(TEXT)
if not OBSCODE:
    ISSUES.append("Beobachtercode fehlt (NB01 Abschnitt 0 / AAVSO-Konto)")
print(f"{len(OBSV)} Beobachtungen; Formatprüfung: " + ("✓ keine Probleme" if not ISSUES else f"{len(ISSUES)} Problem(e)"))
for i in ISSUES[:15]:
    print("  ✗", i)
print("\n".join(TEXT.splitlines()[:12]))

## Schritt 4b — Inhaltsprüfung und Reifegrad

Das Format ist nur die halbe Prüfung. `check_plausibility` prüft den **Inhalt**: Zeit (nicht in der Zukunft), doppelte Zeilen, Helligkeitsbereich des Sterns, Fehlerangaben, Filter/`TRANS`/`OBSTYPE` zur Methode, `CHART`, Streuung und Abweichung des Check-Sterns, Sprünge.
`ERROR` = nicht einreichen (Datei wird nicht geschrieben). `WARN` = vor dem Einreichen bewusst entscheiden; steht im Bericht unter „Offene Punkte“.
Dazu der **Reifegrad**: Funktionen, die noch nicht an echten Daten geprüft wurden, werden genannt (`varphot/maturity.py`). Helligkeitsbereich: `VARPHOT_EXPECTED_RANGE="3.3,4.5"` (sonst grob aus dem Plan).

In [ ]:
from varphot import check_plausibility, maturity_warnings
if os.environ.get("VARPHOT_EXPECTED_RANGE"):
    EXPECTED = tuple(float(x) for x in os.environ["VARPHOT_EXPECTED_RANGE"].split(","))
elif PLAN and PLAN.v_max is not None:
    EXPECTED = (PLAN.v_max - 0.5, PLAN.v_max + 2.0)                    # grob (Amplituden bis ~2 mag); genauer mit VARPHOT_EXPECTED_RANGE
else:
    EXPECTED = None
_ref = {"V": "V", "TG": "V", "B": "B", "TB": "B", "R": "R", "TR": "R"}.get(FILT)
KEXP = float(check[_ref]) if (check and _ref and check.get(_ref) is not None and check[_ref] == check[_ref]) else None
FINDINGS = check_plausibility(OBSV, OBSTYPE, EXPECTED, kmag_expected=KEXP)
if OBSTYPE_UNCONFIRMED:
    from varphot import Finding
    FINDINGS.append(Finding("WARN", "obstype_unbestaetigt", f"OBSTYPE={OBSTYPE} bei Farbkamera/RGB-Filter noch nicht mit der AAVSO geklärt (VARPHOT_OBSTYPE_CONFIRMED=1 nach Klärung)"))
_keys = ["aavso_format", "lightcurve_mono"] + (["transform_fit", "transform_lightcurve"] if MODE_T else []) + (["monorgb"] if PLAN and PLAN.method == "monorgb" else []) \
        + (["cfa_split"] if PLAN and PLAN.method in ("osc", "dslr") else []) + (["obstype_colour"] if COLOUR else [])
MATURE = maturity_warnings(_keys)
ERRORS = [f for f in FINDINGS if f.level == "ERROR"]
WARNS = [f for f in FINDINGS if f.level == "WARN"]
print(f"Inhaltsprüfung: {len(ERRORS)} Fehler, {len(WARNS)} Warnung(en); Reifegrad-Hinweise: {len(MATURE)}")
for f in ERRORS + WARNS:
    print(" ", "✗" if f.level == "ERROR" else "⚠", f.code + ":", f.text)
for m in MATURE:
    print("  ○", m)

## Schritt 5 — Speichern und Zusammenfassung

Die Datei wird nur geschrieben, wenn die Formatprüfung keine Probleme meldet. Einreichen: WebObs → „File Upload“ (Extended Format). Danach den Eintrag im Light Curve Generator ansehen.

In [ ]:
today = datetime.date.today().isoformat()
name = f"aavso_{STAR.replace(' ', '_')}_{FILT}_{today}" + ("_TESTDATEN_NICHT_EINREICHEN" if IS_TESTDATA else "") + ".txt"
if ISSUES or ERRORS:
    print("NICHT gespeichert — erst die Probleme beheben (Format und Inhaltsprüfung).")
else:
    EXPORTS.mkdir(parents=True, exist_ok=True)
    (EXPORTS / name).write_text(TEXT, encoding="utf-8")
    print("gespeichert:", EXPORTS / name)
    sm = pd.Series(dict(Stern=STAR, Filter=FILT, TRANS="YES" if TRANS else "NO", OBSTYPE=OBSTYPE, Punkte=len(OBSV), Zeitraum=f"JD {DATA['jd_mid'].min():.3f} – {DATA['jd_mid'].max():.3f}",
                        mittlerer_Fehler_mag=round(float(DATA["mag_err"].median()), 3), Comps=len(comps), Gaia_Comps=n_gaia, CHART=CHART, Testdaten=IS_TESTDATA))
    display(sm)
    (EXPORTS / (name[:-4] + "_bericht.md")).write_text("# Export " + name + "\n\n" + "\n".join(f"- **{k}**: {v}" for k, v in sm.items()) + "\n\nWarnungen: " + (WARN_CH or "keine") + "\n\n## Offene Punkte (vor dem Einreichen entscheiden)\n" + ("\n".join(f"- {f}" for f in WARNS) or "- keine Warnungen der Inhaltsprüfung") + "\n\n## Reifegrad (nicht an echten Daten geprüft)\n" + ("\n".join(f"- {m}" for m in MATURE) or "- alle benutzten Funktionen an echten Daten geprüft") + "\n", encoding="utf-8")

## Schritt 6 — KI-Zweitlesung des Berichts (optional)

Eine **lokale** KI liest den Berichtstext und die Kopfzeilen der Datei gegen eine feste Prüfliste (Filter/`TRANS`/`OBSTYPE`, Zeitraum, Fehlergröße, `CHART`, Warnungen, Fehlendes). Sie ist **nur zweite Leserin**: Sie ändert nichts und rechnet nichts nach.
Kleine Modelle irren (Fehlalarme und Übersehenes, Probelauf in KONZEPT §11.14) — maßgeblich bleiben die festen Prüfungen oben. Der Text verlässt den Rechner/das Heimnetz nicht (nur private Adressen werden angesprochen).
Weg: stärkerer Rechner im Heimnetz (Opencode-Prüfung, sonst Ollama), sonst lokal (Opencode, dann Ollama) — eingerichtet in NB01/`observer.json`. Abschalten: `VARPHOT_AI_REVIEW=0` oder `VARPHOT_AI=off`.

In [ ]:
from varphot import detect_backends, choose_backend, ai_review
AI_RESULT = None
if os.environ.get("VARPHOT_AI_REVIEW", "1") == "0" or ISSUES or ERRORS:
    print("KI-Zweitlesung übersprungen" + (" (Datei nicht geschrieben)" if (ISSUES or ERRORS) else " (VARPHOT_AI_REVIEW=0)"))
else:
    _bk = choose_backend(detect_backends())
    if _bk is None:
        print("KI-Zweitlesung: kein Weg verfügbar (optional) — siehe Vorbedingungen in NB01")
    else:
        _rep = (EXPORTS / (name[:-4] + "_bericht.md")).read_text(encoding="utf-8")
        _head = "\n".join(TEXT.splitlines()[:12])
        _fnd = "\n".join(str(f) for f in FINDINGS) or "(keine)"
        print(f"KI-Zweitlesung mit {_bk.label()} … (dauert je nach Modell bis ca. 2 Minuten)")
        AI_RESULT = ai_review(_rep + "\n\n## Kopf der Datei\n" + _head + "\n\n## Ergebnis der festen Prüfungen\n" + _fnd, _bk, timeout=300)
        if AI_RESULT.error:
            print("KI-Zweitlesung fehlgeschlagen:", AI_RESULT.error, "— der Bericht bleibt ohne sie gültig")
        else:
            print(AI_RESULT.text)
            with open(EXPORTS / (name[:-4] + "_bericht.md"), "a", encoding="utf-8") as fh:
                fh.write(f"\n## KI-Zweitlesung (unverbindlich; {AI_RESULT.backend}, {AI_RESULT.seconds:g} s)\n\n" + AI_RESULT.text + "\n")